In [21]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


In [22]:
tokenizer("Using a Transformer doesn't require a lot of code.")

{'input_ids': [101, 2478, 1037, 10938, 2121, 2987, 1005, 1056, 5478, 1037, 2843, 1997, 3642, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [23]:
tokenizer.save_pretrained("./tokenizer_save")

('./tokenizer_save/tokenizer_config.json', './tokenizer_save/tokenizer.json')

In [24]:
tokens = tokenizer.tokenize("Let's try to tokenizze this sentence!")

In [25]:
tokens

['let',
 "'",
 's',
 'try',
 'to',
 'token',
 '##iz',
 '##ze',
 'this',
 'sentence',
 '!']

In [26]:
input_ids = tokenizer.convert_tokens_to_ids(tokens)
print(input_ids)

[2292, 1005, 1055, 3046, 2000, 19204, 10993, 4371, 2023, 6251, 999]


In [28]:
cls_id = tokenizer.convert_tokens_to_ids("[CLS]")
sep_id = tokenizer.convert_tokens_to_ids("[SEP]")
final_input_ids = [cls_id] + input_ids + [sep_id]
print(final_input_ids)

[101, 2292, 1005, 1055, 3046, 2000, 19204, 10993, 4371, 2023, 6251, 999, 102]


In [29]:
print(tokenizer.decode(final_input_ids))

[CLS] let ' s try to tokenizze this sentence! [SEP]


In [30]:
decoded_string = tokenizer.decode(final_input_ids)
print(decoded_string)

[CLS] let ' s try to tokenizze this sentence! [SEP]


Testing multiple sequences

In [32]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
sequence = "I've been waiting for a HuggingFace course my whole life."




Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [35]:
tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)
input_ids = torch.tensor([ids])

outputs = model(input_ids)
print(outputs.logits)

tensor([[-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)


In [34]:
tokenized_inputs = tokenizer(sequence, return_tensors="pt")
print(tokenized_inputs["input_ids"])

tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102]])


In [37]:
tokenized_inputs["input_ids"].shape

torch.Size([1, 16])

Applying tokenization individually to each sequence and then batch of sequences.

In [38]:
Sequence1_ids = [[200, 200, 200]]
Sequence2_ids = [[200, 200]]
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id]
]
print(model(torch.tensor(Sequence1_ids)).logits)
print(model(torch.tensor(Sequence2_ids)).logits)
print(model(torch.tensor(batched_ids)).logits)

tensor([[ 1.5694, -1.3895]], grad_fn=<AddmmBackward0>)
tensor([[ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)
tensor([[ 1.5694, -1.3895],
        [ 1.3374, -1.2163]], grad_fn=<AddmmBackward0>)


Getting different results with batches that means padded tokens are affecting the results.

Let's ignore them with the help of attention masks.

In [41]:
attention_mask = [
    [1,1,1],
    [1,1,0],
]
outputs = model(torch.tensor(batched_ids), attention_mask = torch.tensor(attention_mask))
print(outputs.logits)

tensor([[ 1.5694, -1.3895],
        [ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)


To support longer sequences, we need to truncate or pad our inputs to a fixed length.

In [42]:
## Example below
# sequence = sequence[:max_sequence_length]

In [45]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

Sequences = ["I have been waiting for a HuggingFace course my whole life.", "So have I!"]
tokens = tokenizer(Sequences, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
outputs = model(**tokens)
print(outputs)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

SequenceClassifierOutput(loss=None, logits=tensor([[-1.3782,  1.4346],
        [-3.6183,  3.9137]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)
